# 04 - Entrenamiento de RT-DETR para detección de humo y fuego

RT-DETR es un detector basado en transformers con NMS-free y velocidad de
tiempo real. Se entrena con la misma API de Ultralytics que el baseline YOLOv8n,
así que este notebook reusa el pipeline del notebook 02.

In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)
print("Directorio actual:", Path.cwd())

In [ ]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt

print("Dependencias instaladas.")

In [ ]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
    print("GPU:", DEVICE_NAME)
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "cpu"
    print("No se detectó GPU. Faster R-CNN en CPU es inviable para 12 épocas.")

print("Device:", DEVICE)

In [ ]:
# ============================================================
# Montar Google Drive
# ============================================================

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

In [ ]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

PROJECT_DIR = Path("/content") / REPO_NAME

if IN_COLAB:
    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git pull
    else:
        print("Clonando repositorio...")
        %cd /content
        !git clone {REPO_URL}.git
        %cd {PROJECT_DIR}
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:", os.listdir(PROJECT_DIR))

In [ ]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "rtdetr_l.yaml"

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
training_cfg = experiment_config["training"]

print("Experimento:", experiment_name)
print("Modelo:", model_name, "| épocas:", training_cfg["epochs"])

In [ ]:
# ============================================================
# Dataset y YAML para Ultralytics
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"
dataset_root = Path(kagglehub.dataset_download(DATASET_ID))


def find_yolo_dataset_dir(root: Path) -> Path:
    for candidate in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all(
            (candidate / split / kind).exists()
            for split in ["train", "val"]
            for kind in ["images", "labels"]
        ):
            return candidate
    raise FileNotFoundError("No se encontró una estructura YOLO válida.")


DATA_DIR = find_yolo_dataset_dir(dataset_root)
DFIRE_YAML = Path("/content/dfire_colab.yaml")

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        {
            "path": str(DATA_DIR),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "nc": 2,
            "names": {0: "smoke", 1: "fire"},
        },
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print("Dataset:", DATA_DIR)
print(DFIRE_YAML.read_text())

In [ ]:
# ============================================================
# Entrenamiento con RT-DETR
# ============================================================

from ultralytics import RTDETR

project_dir = Path(experiment_config["output"]["project"])
project_dir.mkdir(parents=True, exist_ok=True)
experiment_dir = project_dir / experiment_name

RESUME = (experiment_dir / "weights" / "last.pt").exists()
start_time = time.time()

if RESUME:
    print("Checkpoint encontrado, reanudando entrenamiento.")
    model = RTDETR(str(experiment_dir / "weights" / "last.pt"))
    results = model.train(resume=True)
else:
    print("Entrenamiento desde los pesos preentrenados.")
    model = RTDETR(model_name)
    results = model.train(
        data=str(DFIRE_YAML),
        epochs=training_cfg["epochs"],
        imgsz=training_cfg["imgsz"],
        batch=training_cfg["batch"],
        patience=training_cfg["patience"],
        optimizer=training_cfg["optimizer"],
        lr0=training_cfg["lr0"],
        seed=training_cfg["seed"],
        project=str(project_dir),
        name=experiment_name,
        exist_ok=True,
        plots=True,
    )

train_time_min = (time.time() - start_time) / 60
print(f"Entrenamiento finalizado en {train_time_min:.1f} min")
print("Resultados en:", experiment_dir)

In [ ]:
# ============================================================
# Validación y métricas finales
# ============================================================

best_weights = experiment_dir / "weights" / "best.pt"
model = RTDETR(str(best_weights))

metrics = model.val(data=str(DFIRE_YAML), split="val", plots=True)

print("mAP50   :", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
# ============================================================
# Copiar los artefactos de Ultralytics al repositorio
# ============================================================

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name
REPORTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for filename in [
    "results.csv", "results.png",
    "confusion_matrix.png", "confusion_matrix_normalized.png",
    "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png",
]:
    src_file = experiment_dir / filename
    if src_file.exists():
        shutil.copy(src_file, REPORTS_RESULTS_DIR / filename)
        print("Copiado:", filename)
    else:
        print("No encontrado:", src_file)

with open(REPORTS_RESULTS_DIR / "experiment_config_used.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(experiment_config, file, sort_keys=False, allow_unicode=True)

print("Config usada guardada.")

In [ ]:
# ============================================================
# Exportar metrics_summary.csv con el esquema común
# ============================================================

import numpy as np

from src.reporting.summary import write_metrics_summary

# metrics.box.ap50 y .ap traen una fila por clase QUE TUVO ETIQUETAS, y
# ap_class_index dice a qué clase corresponde cada fila. Indexar por posición
# atribuiría las métricas a la clase equivocada si alguna faltara, así que se
# resuelve por id de clase.
SMOKE, FIRE = 0, 1  # ids del YAML del dataset
fila_de_clase = {int(c): i for i, c in enumerate(metrics.box.ap_class_index)}


def ap_de(vector, clase):
    indice = fila_de_clase.get(clase)
    return float(vector[indice]) if indice is not None else float("nan")


# La velocidad viene en milisegundos por imagen: inferencia más postproceso.
ms_por_imagen = metrics.speed["inference"] + metrics.speed["postprocess"]

precision = float(np.mean(metrics.box.p))
recall = float(np.mean(metrics.box.r))

resumen = {
    "experiment": experiment_name,
    "family": experiment_config["experiment"]["family"],
    "model": model_name,
    "params_M": round(sum(p.numel() for p in model.model.parameters()) / 1e6, 2),
    "epochs": training_cfg["epochs"],
    "imgsz": training_cfg["imgsz"],
    "batch": training_cfg["batch"],
    "train_time_min": round(train_time_min, 2),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50_95": round(float(metrics.box.map), 4),
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1": round(2 * precision * recall / (precision + recall), 4) if (precision + recall) else 0.0,
    "mAP50_smoke": round(ap_de(metrics.box.ap50, SMOKE), 4),
    "mAP50_fire": round(ap_de(metrics.box.ap50, FIRE), 4),
    "mAP50_95_smoke": round(ap_de(metrics.box.ap, SMOKE), 4),
    "mAP50_95_fire": round(ap_de(metrics.box.ap, FIRE), 4),
    "fps": round(1000.0 / ms_por_imagen, 2),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "split": "val",
}

df_resumen = write_metrics_summary(REPORTS_RESULTS_DIR / "metrics_summary.csv", resumen)
display(df_resumen.T.rename(columns={0: "valor"}))

In [ ]:
# ============================================================
# Commit y push de resultados al repositorio
# ============================================================

import subprocess

%cd {PROJECT_DIR}

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

pull_result = subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"], text=True, capture_output=True
)
print(pull_result.stdout, pull_result.stderr)

if pull_result.returncode != 0:
    raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

for path in [f"reports/results/{experiment_name}/", "configs/experiments/rtdetr_l.yaml"]:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)

status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
print(status.stdout)

if not status.stdout.strip():
    print("No hay cambios nuevos para commitear.")
else:
    subprocess.run(
        ["git", "commit", "-m", f"results: update {experiment_name} outputs"], check=True
    )
    print("Commit creado. Para publicarlo: !git push origin main")